In [19]:
import sqlite3
import pandas as pd

con = sqlite3.connect("clinic.db")
df = pd.read_sql("SELECT * FROM patient_visits", con)
df.head()

,patient_id,age,gender,visit_id,visit_date,diagnoses,medicine
0,P041,48,F,V1001,2025-12-09,"hypertension,hypothyroidism",amlodipine
1,P041,48,F,V1001,2025-12-09,"hypertension,hypothyroidism",multivitamin
2,P041,48,F,V1001,2025-12-09,"hypertension,hypothyroidism",levothyroxine
3,P041,48,F,V1171,2025-09-13,"hypertension,hypothyroidism",amlodipine
4,P041,48,F,V1171,2025-09-13,"hypertension,hypothyroidism",levothyroxine


In [20]:
# keepin one row per visit
visits = df.drop_duplicates("visit_id").copy()

# counting distinct medicines per visit
med_counts = df.groupby("visit_id")["medicine"].nunique()
visits["polypharmacy"] = visits["visit_id"].map(med_counts)
visits.head()

,patient_id,age,gender,visit_id,visit_date,diagnoses,medicine,polypharmacy
0,P041,48,F,V1001,2025-12-09,"hypertension,hypothyroidism",amlodipine,3
3,P041,48,F,V1171,2025-09-13,"hypertension,hypothyroidism",amlodipine,3
6,P041,48,F,V1207,2025-06-18,hypertension,amlodipine,1
7,P021,70,F,V1002,2025-05-29,"asthma,type 2 diabetes",salbutamol,3
10,P021,70,F,V1067,2025-10-01,asthma,salbutamol,2


In [21]:
visits = visits.sort_values(["patient_id", "visit_date", "visit_id"])
latest = visits.drop_duplicates("patient_id", keep="last")
latest.head()

,patient_id,age,gender,visit_id,visit_date,diagnoses,medicine,polypharmacy
553,P001,54,M,V5001,2026-01-10,"hypertension,type 2 diabetes",amlodipine,2
240,P002,58,M,V1153,2025-05-26,type 2 diabetes,glimepiride,1
529,P003,62,M,V1210,2025-10-17,"gerd,osteoarthritis",paracetamol,3
348,P004,75,M,V1216,2025-11-30,depression,vitamin d3,2
410,P005,55,F,V1106,2025-10-16,"depression,gerd,osteoarthritis",pantoprazole,3


In [22]:
mean_polypharmacy = latest["polypharmacy"].mean()
print("Mean polypharmacy:", round(mean_polypharmacy, 2))

Mean polypharmacy: 2.38


In [23]:
dx = visits[["patient_id", "visit_id", "diagnoses"]].copy()
dx["diagnosis"] = dx["diagnoses"].str.split(",")
dx = dx.explode("diagnosis")
dx.head()

,patient_id,visit_id,diagnoses,diagnosis
112,P001,V1100,hypothyroidism,hypothyroidism
104,P001,V1091,"gerd,hypertension,hypothyroidism",gerd
104,P001,V1091,"gerd,hypertension,hypothyroidism",hypertension
104,P001,V1091,"gerd,hypertension,hypothyroidism",hypothyroidism
113,P001,V1167,"hypertension,viral fever",hypertension


In [24]:
per_patient = dx.groupby("patient_id")["diagnosis"].nunique()
print("Mean distinct diagnoses:", round(per_patient.mean(), 2))

Mean distinct diagnoses: 2.6


In [25]:
top_diagnoses = dx.groupby("diagnosis")["patient_id"].nunique()
print(top_diagnoses.sort_values(ascending=False).head(5))

diagnosis
viral fever                    25
hypertension                   21
migraine                       20
type 2 diabetes                16
upper respiratory infection    16
Name: patient_id, dtype: int64


In [26]:
meds = df.dropna(subset=["medicine"])
top_medicines = meds.groupby("medicine")["patient_id"].nunique()
print(top_medicines.sort_values(ascending=False).head(5))

medicine
paracetamol    36
vitamin d3     33
amlodipine     21
telmisartan    18
sumatriptan    17
Name: patient_id, dtype: int64


In [27]:
latest = latest.copy()
latest["age_group"] = pd.cut(latest["age"], bins=[-1, 39, 59, 200], labels=["<40", "40-59", "60+"])
latest["high_poly"] = latest["polypharmacy"] >= 5

result = latest.groupby("age_group", observed=False)["high_poly"].mean() * 100
print(result.round(1))

age_group
<40       6.2
40-59     4.5
60+      18.2
Name: high_poly, dtype: float64
